In [4]:
import glob
import numpy as np
import tensorflow as tf
from music21 import converter, instrument, note, chord

In [5]:
midi_files = glob.glob("archive\midi_dataset\midi_dataset\*.mid")

print("midi_files:", len(midi_files))
print(midi_files[:5])

midi_files: 50
['archive\\midi_dataset\\midi_dataset\\x (1).mid', 'archive\\midi_dataset\\midi_dataset\\x (10).mid', 'archive\\midi_dataset\\midi_dataset\\x (11).mid', 'archive\\midi_dataset\\midi_dataset\\x (12).mid', 'archive\\midi_dataset\\midi_dataset\\x (13).mid']


<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\DELL\AppData\Local\Temp\ipykernel_7536\1586753935.py:1: SyntaxWarning: invalid escape sequence '\m'
  midi_files = glob.glob("archive\midi_dataset\midi_dataset\*.mid")


In [6]:
notes = []

for file in midi_files:
    try:
        midi = converter.parse(file)
        print("Parsing %s" % file)

        notes_to_parse = None

        try:  # file has instrument parts
            s2 = instrument.partitionByInstrument(midi)
            notes_to_parse = s2.parts[0].recurse()
        except:  # file has notes in a flat structure
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))
    except Exception as e:
        print("Error parsing %s: %s" % (file, e))
print("Total notes extracted:", len(notes))
print("First 20 notes:", notes[:20])

Parsing archive\midi_dataset\midi_dataset\x (1).mid
Parsing archive\midi_dataset\midi_dataset\x (10).mid
Parsing archive\midi_dataset\midi_dataset\x (11).mid
Parsing archive\midi_dataset\midi_dataset\x (12).mid
Parsing archive\midi_dataset\midi_dataset\x (13).mid
Parsing archive\midi_dataset\midi_dataset\x (14).mid
Parsing archive\midi_dataset\midi_dataset\x (15).mid
Parsing archive\midi_dataset\midi_dataset\x (16).mid
Parsing archive\midi_dataset\midi_dataset\x (17).mid
Parsing archive\midi_dataset\midi_dataset\x (18).mid
Parsing archive\midi_dataset\midi_dataset\x (19).mid
Parsing archive\midi_dataset\midi_dataset\x (2).mid
Parsing archive\midi_dataset\midi_dataset\x (20).mid
Parsing archive\midi_dataset\midi_dataset\x (21).mid
Parsing archive\midi_dataset\midi_dataset\x (22).mid
Parsing archive\midi_dataset\midi_dataset\x (23).mid
Parsing archive\midi_dataset\midi_dataset\x (24).mid
Parsing archive\midi_dataset\midi_dataset\x (25).mid
Parsing archive\midi_dataset\midi_dataset\x (26)

In [7]:
pitchnames = sorted(set(notes))

print("Unique pitch names:", len(pitchnames))

Unique pitch names: 113


In [8]:
notes_to_int ={ note: number for number, note in enumerate(pitchnames)} 

print("vocabulary size:", len(notes_to_int))

vocabulary size: 113


In [9]:
sequence_lenth = 100

network_input = []
network_output = []

for i in range(len(notes) - sequence_lenth):
    sequence_in = notes[i:i + sequence_lenth]
    sequence_out = notes[i + sequence_lenth]

    network_input.append([notes_to_int[n] for n in sequence_in])
    network_output.append(notes_to_int[sequence_out])

print("Input Sequence:", len(network_input))
print("output Sequence:", len(network_output))

Input Sequence: 3792
output Sequence: 3792


In [10]:
n_patterns = len(network_input)

network_input = np.reshape(
    network_input,
    (n_patterns, sequence_lenth, 1)
)

network_input = network_input / float(len(pitchnames))

print(network_input.shape)

(3792, 100, 1)


In [11]:
network_output = tf.keras.utils.to_categorical (network_output)

print(network_output.shape)

(3792, 113)


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

model = Sequential()

model.add(LSTM(
    512,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))
model.add(Dropout(0.3))

model.add(LSTM(512, return_sequences=True))
model.add(Dropout (0.3))

model.add(LSTM(512))
model.add(Dense (256, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(network_output.shape[1], activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer = "adam")

model.summary()

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 512)       │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100, 512)       │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 512)            │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 113)            │        29,041 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,411,441 (20.64 MB)

 Trainable params: 5,411,441 (20.64 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor = "loss",
    verbose = 1,
    save_best_only=True,
    mode="min" )

history = model.fit(
    network_input,
    network_output,
    epochs = 5,
    batch_size=64,
    callbacks=[checkpoint] )

Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - loss: 4.4229 
Epoch 1: loss improved from None to 4.29369, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 578s 9s/step - loss: 4.2937
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - loss: 4.0584
Epoch 2: loss improved from 4.29369 to 4.03390, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 562s 9s/step - loss: 4.0339
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - loss: 3.9192
Epoch 3: loss improved from 4.03390 to 3.84538, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 563s 9s/step - loss: 3.8454
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - loss: 3.7823
Epoch 4: loss improved from 3.84538 to 3.78999, saving model to best_model.keras

Epoch 4: finished saving model to best_model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 543s 9s/step - loss: 3.790

In [14]:
from tensorflow.keras.models import load_model
import random

model=load_model("best_model.keras")

start=random.randint(0, len(network_input)-1)

int_to_note={number: note for number, note in enumerate(pitchnames)}

pattern=network_input [start]
prediction_output = []

for _ in range(500):
    prediction_input=np.reshape(pattern, (1, len(pattern), 1))
    prediction=model.predict (prediction_input, verbose=8)

    index = np.argmax(prediction)
    result=int_to_note[index]
    prediction_output.append(result)

    pattern=np.append(pattern, index/float(len(pitchnames)))
    pattern=pattern[1:]

In [15]:
from music21 import stream

offset = 0
output_notes = []

for pattern in prediction_output:
    if "." in pattern or pattern.isdigit():
        notes_in_chord=pattern.split(".")
        chord_notes = []

        for current_note in notes_in_chord:
            new_note=note.Note(int(current_note))
            new_note.storedInstrument=instrument.Piano()
            chord_notes.append(new_note)
        new_chord=chord. Chord(chard_notes)
        new_chord.offset = offset
        output_notes.append(new_chord)
    else:
        new_note=note.Note(pattern)
        new_note.offset = offset

    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generated_music.mid")

print("Music generated successfully: generated_music.mid")

Music generated successfully: generated_music.mid


In [16]:
import pygame
import time


pygame.init()
pygame.mixer.init()
pygame.mixer.music.load("generated_music.mid")
pygame.mixer.music.play()

print("Playing generated_music.mid...")


while pygame.mixer.music.get_busy():
    time.sleep(1)

print("Finished playing.") 

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.10)
Hello from the pygame community. https://www.pygame.org/contribute.html
Playing generated_music.mid...
Finished playing.
